# Prompt Evaluation & Optimization with Vertex AI SDK

# ----------------------------------------------------

This notebook demonstrates how to perform prompt evaluation and optimization
using Vertex AI Generative AI Evaluation SDK.
It includes: dataset prep, metric setup, evaluation run, optimization loop.

## Setup and installation


In [1]:
from google.colab import auth

# Authenticate your Google Cloud account
auth.authenticate_user()
print('Google Cloud authentication complete.')

Google Cloud authentication complete.


In [2]:
!pip install --upgrade google-cloud-aiplatform[evaluation] --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 18.4 MB/s eta 0:00:00


In [3]:
import os
import pandas as pd
from google.cloud import aiplatform
import vertexai
from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate

In [4]:
PROJECT_ID = "jrproject-402905"
LOCATION = "us-central1"
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Initialized Vertex AI for project {PROJECT_ID}, location {LOCATION}")

Initialized Vertex AI for project jrproject-402905, location us-central1


## Prepare evaluation dataset


In [5]:
df = pd.DataFrame({
    'prompt': ["Summarize the benefits of AI in healthcare.", "Explain cloud computing in simple terms."],
    'response': ["AI improves diagnostics and personalized care.", "Cloud computing means using remote servers for storage and processing."],
    'reference': ["AI in healthcare aids diagnosis and treatment personalization.", "Cloud computing uses internet-based servers instead of local ones."]
})

df.to_json("eval_dataset.jsonl", orient="records", lines=True)
print("Evaluation dataset created and saved as eval_dataset.jsonl")


Evaluation dataset created and saved as eval_dataset.jsonl


## Define evaluation metrics


In [6]:
readability_metric = PointwiseMetric(
    metric="readability_grade_level",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "grade_level": (
                "Estimate the U.S. grade-level readability of the response. Lower is simpler, higher is complex."
            ),
            "conciseness": (
                "Evaluate how concise the response is: avoids unnecessary words while preserving meaning."
            )
        },
        rating_rubric={
            "1": "Excellent readability and conciseness.",
            "0": "Average readability and conciseness.",
            "-1": "Poor readability or verbosity."
        },
        input_variables=["prompt", "response", "reference"]
    )
)
print("Custom readability metric defined.")


Custom readability metric defined.


## Run evaluation

In [9]:
eval_task = EvalTask(
    dataset="eval_dataset.jsonl",
    metrics=[readability_metric, "bleu", "rouge_l_sum"],
    experiment="prompt-eval-experiment12"
)

print("Running evaluation task...")
eval_result = eval_task.evaluate()
print("Evaluation completed.")

Running evaluation task...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 6 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 6/6 [00:20<00:00,  3.47s/it]
INFO:vertexai.evaluation._evaluation:All 6 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:20.84378779600047 seconds


Evaluation completed.


## View evaluation results


In [16]:
eval_result.summary_metrics

{'row_count': 2,
 'readability_grade_level/mean': np.float64(1.0),
 'readability_grade_level/std': 0.0,
 'bleu/mean': np.float64(0.0731709925),
 'bleu/std': 0.011625772399191908,
 'rouge_l_sum/mean': np.float64(0.29285715),
 'rouge_l_sum/std': 0.010101515343996672}

In [17]:
per_instance = eval_result.metrics_table
per_instance

,prompt,response,reference,readability_grade_level/explanation,readability_grade_level/score,bleu/score,rouge_l_sum/score
0,Summarize the benefits of AI in healthcare.,AI improves diagnostics and personalized care.,AI in healthcare aids diagnosis and treatment ...,"The response is highly concise, effectively su...",1.0,0.064950,0.285714
1,Explain cloud computing in simple terms.,Cloud computing means using remote servers for...,Cloud computing uses internet-based servers in...,The response is extremely concise and uses sim...,1.0,0.081392,0.300000


## New Prompt Evaluation: Summarization Task

This section demonstrates a new prompt evaluation using a summarization task, where the model generates the responses based on the provided prompts. We will evaluate a broader set of metrics including ROUGE-1, ROUGE-2, additional readability scores (ARI, Flesch-Kincaid), and a custom LLM-as-judge metric for conciseness.

In [18]:
# Create a new dataset for summarization
summarization_df = pd.DataFrame({
    'prompt': [
        "Summarize the key points of this article: 'Artificial intelligence (AI) is intelligence demonstrated by machines, unlike the natural intelligence displayed by humans and animals. Leading AI textbooks define the field as the study of 'intelligent agents': any device that perceives its environment and takes actions that maximize its chance of successfully achieving its goals. Colloquially, the term 'artificial intelligence' is often used to describe machines (or computers) that mimic 'cognitive' functions that humans associate with the human mind, such as 'learning' and 'problem-solving'. Such functions are typically associated with other human traits, such as emotion; however, such connections are not necessary for a functional system that produces intelligent behavior.'",
        "Summarize the plot of Romeo and Juliet in one sentence."
    ],
    'reference': [
        "AI is machine intelligence that mimics human cognitive functions like learning and problem-solving, defined as the study of intelligent agents that act to achieve goals.",
        "Romeo and Juliet is a tragedy about two young star-crossed lovers whose deaths ultimately reconcile their feuding families."
    ]
})

summarization_df.to_json("summarization_dataset.jsonl", orient="records", lines=True)
print("Summarization dataset created and saved as summarization_dataset.jsonl")

Summarization dataset created and saved as summarization_dataset.jsonl


In [19]:
# Define LLM-as-judge metric for Conciseness
conciseness_metric = PointwiseMetric(
    metric="llm_conciseness",
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            "conciseness": (
                "Evaluate how concise the response is, avoiding unnecessary words while preserving meaning."
            )
        },
        rating_rubric={
            "1": "Excellent conciseness. The response is brief, to the point, and doesn't contain any superfluous information.",
            "0": "Average conciseness. The response is generally concise but could be slightly shorter without losing meaning, or has minor redundancy.",
            "-1": "Poor conciseness. The response is verbose, contains irrelevant information, or could be significantly shortened."
        },
        input_variables=["prompt", "response", "reference"]
    )
)
print("Custom conciseness metric defined.")

Custom conciseness metric defined.


In [26]:
# Run evaluation for summarization task, with model generating responses
summarization_eval_task = EvalTask(
    dataset="summarization_dataset.jsonl",
    metrics=[
        "bleu",
        "rouge_1",
        "rouge_2",
        "rouge_l_sum",
        conciseness_metric
    ],
    experiment="summarization-eval-experiment"
)

print("Running summarization evaluation task...")
summarization_eval_result = summarization_eval_task.evaluate(
    model="gemini-2.5-flash", # Model to generate responses
    prompt_template="{prompt}" # Template for prompt sent to the model
)
print("Summarization evaluation completed.")

Running summarization evaluation task...


INFO:vertexai.evaluation.eval_task:Logging Eval Experiment metadata: {'prompt_template': '{prompt}', 'model_name': 'gemini-2.5-flash'}
INFO:vertexai.evaluation._evaluation:Assembling prompts from the `prompt_template`. The `prompt` column in the `EvalResult.metrics_table` has the assembled prompts used for model response generation.
INFO:vertexai.evaluation._evaluation:Generating a total of 2 responses from Gemini model gemini-2.5-flash using genai module.
100%|██████████| 2/2 [00:05<00:00,  2.54s/it]
INFO:vertexai.evaluation._evaluation:All 2 responses are successfully generated from Gemini model gemini-2.5-flash using genai module.
INFO:vertexai.evaluation._evaluation:Multithreaded Batch Inference took: 5.271983495000313 seconds.
INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 10 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 10/10 [00:09<00:00,  1.09it/s]
INFO:vertexai.evaluation._evaluation:All 10 metric requests are successfully computed.
IN

Summarization evaluation completed.


### View Summarization Evaluation Results

In [28]:
print("Summary Metrics for Summarization Task:")
print(summarization_eval_result.summary_metrics)

Summary Metrics for Summarization Task:
{'row_count': 2, 'bleu/mean': np.float64(0.07878718), 'bleu/std': 0.014242459933845698, 'rouge_1/mean': np.float64(0.38584071), 'rouge_1/std': 0.020024259951573767, 'rouge_2/mean': np.float64(0.187055475), 'rouge_2/std': 0.0412395210679034, 'rouge_l_sum/mean': np.float64(0.27699115), 'rouge_l_sum/std': 0.10888192851270133, 'llm_conciseness/mean': np.float64(1.0), 'llm_conciseness/std': 0.0}


In [29]:
print("Per-instance Metrics for Summarization Task:")
summarization_per_instance = summarization_eval_result.metrics_table
display(summarization_per_instance)

Per-instance Metrics for Summarization Task:


,prompt,reference,response,bleu/score,rouge_1/score,rouge_2/score,rouge_l_sum/score,llm_conciseness/explanation,llm_conciseness/score
0,Summarize the key points of this article: 'Art...,AI is machine intelligence that mimics human c...,This article defines Artificial Intelligence (...,0.088858,0.371681,0.216216,0.353982,"The response is brief and to the point, clearl...",1.0
1,Summarize the plot of Romeo and Juliet in one ...,Romeo and Juliet is a tragedy about two young ...,"Two young lovers from rival families, Romeo an...",0.068716,0.400000,0.157895,0.200000,"The response is brief, to the point, and effec...",1.0
